In [159]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu

np.random.seed(42)
import random
random.seed(42)

In [160]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
import os
os.environ["SCIPY_ARRAY_API"] = "1"
from imblearn.over_sampling import RandomOverSampler

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

torch.cuda.is_available()

True

In [ ]:
file_name = 'merged_EAE_CD4_CV0'
features = pd.read_csv(f'{file_name}.csv')
features.head(5)

,feature_gex1,feature_gex2,feature_gex3,feature_gex4,feature_gex5,feature_gex6,feature_gex7,feature_gex8,feature_gex9,feature_gex10,...,feature_tcr1313,feature_tcr1314,feature_tcr1315,feature_tcr1316,feature_tcr1317,CV_score_1,label_tissue,label_cell_type,label_state,label_set
0,0.625604,1.101336,-0.872533,0.362678,0.348406,1.691799,-1.513119,1.556052,-0.220529,1.428859,...,-0.050723,-0.104547,0.0,-0.358447,-1.058772,0.151809,CNS,CD4,Activation,train
1,-1.633328,-0.016522,1.186384,-0.566553,-0.412338,1.130019,0.333264,0.921487,1.483764,0.552362,...,-0.050723,-0.104547,0.0,0.607462,-0.088085,-0.050504,CNS,NaN,Activation,train
2,-0.022191,1.078571,-0.943021,-0.652273,1.530086,0.987665,-0.131203,1.342826,-0.098099,-0.210875,...,-0.050723,-0.104547,0.0,0.607462,0.882603,-1.314188,CNS,Treg,Activation,train
3,0.963593,1.206184,-0.711800,0.146631,0.559764,1.607911,-1.158318,1.712503,-0.349028,1.339895,...,-0.050723,-0.104547,0.0,-0.358447,-1.058772,0.151809,CNS,CD4,Activation,train
4,2.107041,-0.101425,-0.238915,0.194247,-0.041171,0.060662,0.256297,0.088335,2.699889,0.419907,...,-0.050723,-0.104547,0.0,-0.358447,-0.088085,0.336447,CNS,NaN,Activation,train


In [162]:
all_tcr_features = False

In [163]:
features['label_tissue'] = features['label_tissue'].apply(lambda x: 'CNS' if x == 'CNS' else 'Not CNS')
features['label_tissue'].value_counts()

label_tissue
CNS        13676
Not CNS    10094
Name: count, dtype: int64

In [164]:
features['label_state'].value_counts()
activation_mask = (features['label_state'] == 'Exhaust') & (features['label_set'] == 'test')
cv_scores_activation = features.loc[activation_mask, 'CV_score_2']
print(len(cv_scores_activation))

# Plot distribution of CV_score_2 by label_tissue (CNS vs Not CNS)
plt.figure(figsize=(7, 4))
for label, group in features[activation_mask].groupby('label_tissue'):
    plt.hist(group['CV_score_2'].dropna(), bins=30, alpha=0.7, label=label, edgecolor='k')
plt.title("Distribution of CV_score_2 for CNS vs Not CNS")
plt.xlabel("CV_score_2")
plt.ylabel("Count")
plt.legend()
plt.show()


KeyError: 'CV_score_2'

In [ ]:
required_columns = {"label_tissue", "label_set"}
missing_required = required_columns - set(features.columns)
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

clean_df = features.dropna(subset=required_columns).copy()
dropped_rows = len(features) - len(clean_df)
if dropped_rows > 0:
    print(f"Dropped {dropped_rows} rows with missing 'label_tissue' or 'label_set'.")

label_series = clean_df['label_tissue']
label_encoder = {label: idx for idx, label in enumerate(sorted(label_series.unique()))}
num_classes = len(label_encoder)
if num_classes != 2:
    raise ValueError(f"Expected exactly two classes for binary classification, found {num_classes}.")

labels = label_series.map(label_encoder).astype(np.float32).values

set_series = clean_df['label_set'].str.lower()
allowed_sets = {"train", "test"}
if not set(set_series.unique()).issubset(allowed_sets):
    raise ValueError("'label_set' must contain only 'train' and 'test' values.")

train_mask = set_series == 'train'
test_mask = set_series == 'test'
if train_mask.sum() == 0 or test_mask.sum() == 0:
    raise ValueError("Both train and test splits must contain at least one sample.")

train_indices = np.where(train_mask)[0]
test_indices = np.where(test_mask)[0]

feature_sets = {
    "feature_gex": [col for col in clean_df.columns if col.startswith('feature_gex')],
}

# feature_sets["feature_gex_tcr"] = sorted({*feature_sets["feature_gex"], *feature_sets["feature_tcr"]})


In [ ]:
from typing import Any

if all_tcr_features:
    tcr_prefix = "feature_tcr"
    tcr_columns = [col for col in clean_df.columns if col.startswith(tcr_prefix)]
else:
    tcr_prefix = "CV_score_"
    tcr_columns = [col for col in clean_df.columns if col.startswith(tcr_prefix)]

if not tcr_columns:
    raise ValueError(f"No columns found with prefix '{tcr_prefix}'.")

feature_sets["feature_tcr"] = tcr_columns
feature_sets["feature_gex_tcr"] = sorted({*feature_sets["feature_gex"], *feature_sets["feature_tcr"]})

print(f"TCR feature prefix: {tcr_prefix} (all_tcr_features={all_tcr_features})")

for key, cols in feature_sets.items():
    print(f"{key}: {len(cols)} columns")



TCR feature prefix: CV_score_ (all_tcr_features=False)
feature_gex: 50 columns
feature_tcr: 1 columns
feature_gex_tcr: 51 columns


In [ ]:
from IPython.display import display

if 'label_encoded' not in clean_df.columns:
    clean_df['label_encoded'] = clean_df['label_tissue'].map(label_encoder).astype(np.float32)

cell_type_categories = sorted(clean_df['label_cell_type'].dropna().unique())
state_categories = sorted(clean_df['label_state'].dropna().unique())


class BinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def build_loaders(df, selected_columns, batch_size_train=64, batch_size_test=128):
    if len(selected_columns) == 0:
        raise ValueError("Selected feature list is empty.")
    if df.empty:
        raise ValueError("No rows available after filtering.")

    set_series = df['label_set'].str.lower()
    train_mask = set_series == 'train'
    test_mask = set_series == 'test'

    if train_mask.sum() == 0 or test_mask.sum() == 0:
        raise ValueError("Both train and test splits must contain at least one sample.")

    y = df['label_encoded'].astype(np.float32).values
    y_train = y[train_mask]
    y_test = y[test_mask]

    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        raise ValueError("Train and test splits must each contain both classes.")

    X = df[selected_columns].astype(np.float32).values
    X_train = np.ascontiguousarray(X[train_mask])
    X_test = np.ascontiguousarray(X[test_mask])

    y_train = np.ascontiguousarray(y_train)
    y_test = np.ascontiguousarray(y_test)

    ros = RandomOverSampler(random_state=42)
    X_train, y_train = ros.fit_resample(X_train, y_train)

    X_train = np.ascontiguousarray(X_train, dtype=np.float32)
    y_train = np.ascontiguousarray(y_train.astype(np.float32))

    train_dataset = TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train),
    )
    test_dataset = TensorDataset(
        torch.from_numpy(X_test),
        torch.from_numpy(y_test),
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size_test, shuffle=False)

    return train_loader, test_loader, X.shape[1]


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            preds = torch.sigmoid(logits) > 0.5
            correct += (preds.int() == yb.int()).sum().item()
            count += xb.size(0)
    avg_loss = total_loss / max(count, 1)
    accuracy = correct / max(count, 1)
    return avg_loss, accuracy


def train_single_run(run_label, df, selected_columns, device, num_epochs, learning_rate):
    try:
        train_loader, test_loader, input_dim = build_loaders(df, selected_columns)
    except ValueError as exc:
        print(f"Skipping {run_label}: {exc}")
        return None

    model = BinaryClassifier(input_dim=input_dim, hidden_dim=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    log_epochs = {1, num_epochs}
    if num_epochs >= 5:
        log_epochs.update(range(5, num_epochs + 1, 5))

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        train_loss = running_loss / max(len(train_loader.dataset), 1)
        test_loss, test_acc = evaluate(model, test_loader, device)

        if epoch in log_epochs:
            print(
                f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | "
                f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.3f}"
            )

    final_train_loss, _ = evaluate(model, train_loader, device)
    final_test_loss, final_test_acc = evaluate(model, test_loader, device)

    return {
        "train_loss": final_train_loss,
        "test_loss": final_test_loss,
        "test_acc": round(final_test_acc, 3),
    }


def run_scope(scope_name, df, feature_sets, device, num_epochs, learning_rate):
    scope_results = []
    print(f"\n=== Scope: {scope_name} | samples={len(df)} ===")
    for feature_name, selected_columns in feature_sets.items():
        run_label = f"{scope_name} | {feature_name}"
        print(f"\n--- Training on {feature_name} ({len(selected_columns)} features) ---")
        result = train_single_run(run_label, df, selected_columns, device, num_epochs, learning_rate)
        if result is None:
            continue
        cns_ratio = df['label_tissue'].eq('CNS').mean() if len(df) > 0 else np.nan
        result.update({
            "scope": scope_name,
            "feature_set": feature_name,
            "num_features": len(selected_columns),
            "samples": len(df),
            "cns_pct_unbalanced": round(cns_ratio * 100, 3) if not np.isnan(cns_ratio) else np.nan,
        })
        scope_results.append(result)
    return scope_results



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()
num_epochs = 20
learning_rate = 1e-3

all_results = []

# Overall dataset
all_results.extend(run_scope("all_samples", clean_df, feature_sets, device, num_epochs, learning_rate))

# Subsets by label_cell_type
for cell_type in cell_type_categories:
    subset_df = clean_df[clean_df['label_cell_type'] == cell_type]
    scope_name = f"label_cell_type={cell_type}"
    all_results.extend(run_scope(scope_name, subset_df, feature_sets, device, num_epochs, learning_rate))

# Subsets by label_state
for state in state_categories:
    subset_df = clean_df[clean_df['label_state'] == state]
    scope_name = f"label_state={state}"
    all_results.extend(run_scope(scope_name, subset_df, feature_sets, device, num_epochs, learning_rate))

if all_results:
    summary_df = pd.DataFrame(all_results)
    summary_df = summary_df[
        ['scope', 'feature_set', 'num_features', 'samples', 'cns_pct_unbalanced', 'test_acc']
    ].sort_values(['scope', 'feature_set']).reset_index(drop=True)

    display_summary = summary_df.copy()
    display_summary['test_acc'] = display_summary['test_acc'].round(3)
    for col in ('feature_set', 'scope'):
        display_summary[col] = (
            display_summary[col]
            .str.replace('label_state=', '', regex=False)
            .str.replace('label_cell_type=', '', regex=False)
        )

    display(display_summary)

    best_by_scope = (
        display_summary.loc[display_summary.groupby('scope')['test_acc'].idxmax(), ['scope', 'feature_set', 'test_acc']]
        .rename(columns={'feature_set': 'best_feature_set', 'test_acc': 'best_test_acc'})
        .reset_index(drop=True)
    )
    best_by_scope['best_test_acc'] = best_by_scope['best_test_acc'].round(3)
    print("\n=== Best performing feature set per scope ===")
    display(best_by_scope)

    accuracy_pivot = display_summary.pivot_table(index='scope', columns='feature_set', values='test_acc').round(3)
    print("\n=== Accuracy comparison (test_acc) ===")
    display(accuracy_pivot)

    if not accuracy_pivot.empty and "feature_gex_tcr" in accuracy_pivot.columns:
        cns_pct_by_scope = summary_df.groupby('scope')['cns_pct_unbalanced'].first()
        comparison_df = (
            accuracy_pivot
            .assign(
                combined_minus_gex=accuracy_pivot["feature_gex_tcr"] - accuracy_pivot.get("feature_gex", 0.0),
                combined_minus_tcr=accuracy_pivot["feature_gex_tcr"] - accuracy_pivot.get("feature_tcr", 0.0),
                cns_pct_unbalanced=cns_pct_by_scope.reindex(accuracy_pivot.index)
            )
            .sort_values("feature_gex_tcr", ascending=False)
        )
        comparison_df['cns_pct_unbalanced'] = comparison_df['cns_pct_unbalanced'].round(3)
        comparison_df = comparison_df.round(3)
        print("\n=== Combined vs individual models (test_acc deltas) ===")
        display(comparison_df)





=== Scope: all_samples | samples=23770 ===

--- Training on feature_gex (50 features) ---
Epoch 01 | Train Loss: 0.5550 | Test Loss: 0.3585 | Test Acc: 0.867
Epoch 05 | Train Loss: 0.2809 | Test Loss: 0.2968 | Test Acc: 0.885
Epoch 10 | Train Loss: 0.2429 | Test Loss: 0.3305 | Test Acc: 0.873
Epoch 15 | Train Loss: 0.2268 | Test Loss: 0.3652 | Test Acc: 0.857
Epoch 20 | Train Loss: 0.2117 | Test Loss: 0.3818 | Test Acc: 0.857

--- Training on feature_tcr (1 features) ---
Epoch 01 | Train Loss: 0.6939 | Test Loss: 0.6897 | Test Acc: 0.529
Epoch 05 | Train Loss: 0.6797 | Test Loss: 0.6948 | Test Acc: 0.528
Epoch 10 | Train Loss: 0.6767 | Test Loss: 0.6887 | Test Acc: 0.554
Epoch 15 | Train Loss: 0.6760 | Test Loss: 0.7006 | Test Acc: 0.508
Epoch 20 | Train Loss: 0.6738 | Test Loss: 0.6984 | Test Acc: 0.532

--- Training on feature_gex_tcr (51 features) ---
Epoch 01 | Train Loss: 0.5441 | Test Loss: 0.3796 | Test Acc: 0.861
Epoch 05 | Train Loss: 0.2752 | Test Loss: 0.3207 | Test Acc: 0.

,scope,feature_set,num_features,samples,test_acc
0,all_samples,feature_gex,50,23770,0.857
1,all_samples,feature_gex_tcr,51,23770,0.857
2,all_samples,feature_tcr,1,23770,0.532
3,CD4,feature_gex,50,7135,0.661
4,CD4,feature_gex_tcr,51,7135,0.661
5,CD4,feature_tcr,1,7135,0.581
6,Th17,feature_gex,50,1468,0.892
7,Th17,feature_gex_tcr,51,1468,0.914
8,Th17,feature_tcr,1,1468,0.396
9,Treg,feature_gex,50,3248,0.793



=== Best performing feature set per scope ===


,scope,best_feature_set,best_test_acc
0,Activation,feature_gex_tcr,0.925
1,CD4,feature_gex,0.661
2,Exhaust,feature_gex,0.826
3,IFN_stim,feature_gex,0.886
4,Mem_Naive,feature_gex_tcr,0.707
5,Th17,feature_gex_tcr,0.914
6,Treg,feature_gex,0.793
7,all_samples,feature_gex,0.857



=== Accuracy comparison (test_acc) ===


feature_set,feature_gex,feature_gex_tcr,feature_tcr
scope,,,
Activation,0.916,0.925,0.498
CD4,0.661,0.661,0.581
Exhaust,0.826,0.826,0.727
IFN_stim,0.886,0.864,0.455
Mem_Naive,0.621,0.707,0.707
Th17,0.892,0.914,0.396
Treg,0.793,0.782,0.540
all_samples,0.857,0.857,0.532



=== Combined vs individual models (test_acc deltas) ===


feature_set,feature_gex,feature_gex_tcr,feature_tcr,combined_minus_gex,combined_minus_tcr
scope,,,,,
Activation,0.916,0.925,0.498,0.009,0.427
Th17,0.892,0.914,0.396,0.022,0.518
IFN_stim,0.886,0.864,0.455,-0.022,0.409
all_samples,0.857,0.857,0.532,0.000,0.325
Exhaust,0.826,0.826,0.727,0.000,0.099
Treg,0.793,0.782,0.540,-0.011,0.242
Mem_Naive,0.621,0.707,0.707,0.086,0.000
CD4,0.661,0.661,0.581,0.000,0.080


In [ ]:
# Placeholder cell intentionally left blank; preprocessing is handled upstream.
comparison_df.to_csv(f'{file_name}_allTCR_{all_tcr_features}_comparison_df.csv', index=False)
